# 00_theMonster: exploring the Monster reference catalog through the Butler

Goals:
1. find the `the_monster_*` dataset type and the collections that hold it (`main` / `dp2` butlers at CC-IN2P3);
2. read the HTM7 shards covering the COSMOS field;
3. inventory the columns (units, flux/error columns per band, flags);
4. basic sanity plots (sky, magnitudes, colours);
5. optional: cross-match with a step-1 product (`single_visit_star`) to prepare the 05_AnaStep1 series.

Helpers come from `src/rubincc/monster.py`, adapted from `LSSTDESC/spesn` (`spesn/io.py`, kept untouched in `src/spesn`).

Run with the `lsst` kernel at CC-IN2P3 (valid token needed: `lsstfr login`).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# make the local package src/rubincc importable (this notebook lives in notebooks/05_AnaStep1)
SRC = Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rubincc import monster as mon

NB_TAG = '00_theMonster'
DATA_DIR = Path(f'data_{NB_TAG}')
FIG_DIR = Path(f'figs_{NB_TAG}')
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.labelsize'] = 'x-large'
plt.rcParams['axes.titlesize'] = 'x-large'
plt.rcParams['xtick.labelsize'] = 'x-large'
plt.rcParams['ytick.labelsize'] = 'x-large'


def savefig(fig, name):
    # every figure is saved as PDF and PNG
    for ext in ('pdf', 'png'):
        fig.savefig(FIG_DIR / f'{NB_TAG}_{name}.{ext}', bbox_inches='tight', dpi=150)

In [ ]:
REPO = 'main'  # 'main' or 'dp2'
MONSTER_DATASET = mon.DEFAULT_MONSTER_DATASET

# COSMOS field centre (deg) and cone radius (deg)
RA0, DEC0 = 150.1191, 2.2058
RADIUS = 0.5

## 1. Butler: dataset type and collections

In [ ]:
butler = mon.get_butler(REPO)
print(mon.find_monster_dataset_types(butler))

In [ ]:
dstype = butler.registry.getDatasetType(MONSTER_DATASET)
print('name         :', dstype.name)
print('dimensions   :', dstype.dimensions)
print('storageClass :', dstype.storageClass_name)

In [ ]:
candidates = []
for glob in ('*refcat*', '*defaults*'):
    found = mon.find_collections_with_dataset(butler, MONSTER_DATASET, collection_glob=glob, max_collections=10)
    for name in found:
        if name not in candidates:
            candidates.append(name)
print(candidates)

# if empty, set the collection by hand
MONSTER_COLLECTIONS = candidates[0] if candidates else None
print('using collections =', MONSTER_COLLECTIONS)

## 2. Raw shard: columns, dtypes, units

Check here, before trusting any conversion, the unit of `coord_ra` / `coord_dec` and of the flux columns.

In [ ]:
htm_ids = mon.htm_ids_for_cone(RA0, DEC0, RADIUS)
print('HTM7 ids:', htm_ids)

raw = mon.load_monster_shards(butler, htm_ids[:1], dataset=MONSTER_DATASET, collections=MONSTER_COLLECTIONS)
print(type(raw), len(raw), 'rows,', len(raw.colnames), 'columns')

inventory = pd.DataFrame({
    'column': raw.colnames,
    'dtype': [str(raw[n].dtype) for n in raw.colnames],
    'unit': [str(raw[n].unit) for n in raw.colnames],
})
inventory.to_csv(DATA_DIR / f'{NB_TAG}_column_inventory.csv', index=False)
inventory

In [ ]:
print('coord_ra  :', raw['coord_ra'].unit, np.asarray(raw['coord_ra'][:3]))
print('coord_dec :', raw['coord_dec'].unit, np.asarray(raw['coord_dec'][:3]))
print('degrees after conversion:', mon.to_degrees(raw['coord_ra'])[:3], mon.to_degrees(raw['coord_dec'])[:3])

## 3. Cone around COSMOS

In [ ]:
df = mon.load_monster_cone(butler, RA0, DEC0, RADIUS, dataset=MONSTER_DATASET, collections=MONSTER_COLLECTIONS)
print(df.shape)
try:
    df.to_parquet(DATA_DIR / f'{NB_TAG}_cosmos_cone.parquet')
except Exception as exc:
    print('parquet not written:', exc)
df.head()

In [ ]:
fc = mon.discover_flux_columns(df.columns)
print(fc['prefix'].unique())
fc.pivot_table(index='prefix', columns='band', values='column', aggfunc='count').fillna(0).astype(int)

In [ ]:
# valid (finite, >0) flux counts per flux column
flux_cols = fc[fc['kind'] == 'flux']['column'].tolist()
valid = pd.Series({c: int((df[c] > 0).sum()) for c in flux_cols}, name='n_valid')
valid

## 4. Sanity plots

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(df['ra_deg'], df['dec_deg'], s=1, alpha=0.3)
ax.set_xlabel('RA (deg)')
ax.set_ylabel('Dec (deg)')
ax.set_title(f'the_monster around COSMOS (N={len(df)})')
ax.invert_xaxis()
ax.set_aspect(1.0 / np.cos(np.deg2rad(DEC0)))
savefig(fig, 'sky')

In [ ]:
# choose the flux prefix to plot (see the pivot table above)
PREFIX = fc['prefix'].unique()[0]
print('PREFIX =', PREFIX)

bands = [b for b in 'ugrizy' if f'{PREFIX}_{b}_flux' in df.columns]
fig, ax = plt.subplots()
for b in bands:
    mag = mon.flux_njy_to_abmag(df[f'{PREFIX}_{b}_flux'])
    ax.hist(mag[np.isfinite(mag)], bins=np.arange(12, 30, 0.25), histtype='step', lw=2, label=b)
ax.set_xlabel('AB magnitude (assuming flux in nJy)')
ax.set_ylabel('N')
ax.legend()
savefig(fig, 'mag_hist')

In [ ]:
if all(b in bands for b in 'gri'):
    g = mon.flux_njy_to_abmag(df[f'{PREFIX}_g_flux'])
    r = mon.flux_njy_to_abmag(df[f'{PREFIX}_r_flux'])
    i = mon.flux_njy_to_abmag(df[f'{PREFIX}_i_flux'])
    ok = np.isfinite(g) & np.isfinite(r) & np.isfinite(i)
    fig, ax = plt.subplots()
    hb = ax.hexbin(g[ok] - r[ok], r[ok] - i[ok], gridsize=80, bins='log', extent=(-0.5, 2.5, -0.5, 2.5))
    fig.colorbar(hb, ax=ax, label='log10 N')
    ax.set_xlabel('g - r')
    ax.set_ylabel('r - i')
    savefig(fig, 'colour_colour')
else:
    print('g, r, i not all available for this prefix:', bands)

## 5. Optional: cross-match with a step-1 product

Fill `RUN_COLLECTION` (output collection of your BPS run) and `VISIT`, then run. The dataset `single_visit_star` is the consolidated per-visit source table produced at the end of stage 1 (`consolidateSingleVisitStar` in the w_2026_43 pipeline). Column names and the flux unit (nJy expected) must be checked first.

In [ ]:
RUN_COLLECTION = None  # e.g. the output collection of the 2026-10-27 stage1/stage2 runs
VISIT = None
BAND = 'g'

if RUN_COLLECTION is None or VISIT is None:
    print('Set RUN_COLLECTION and VISIT to run the cross-check.')
else:
    run_butler = mon.get_butler(REPO, collections=RUN_COLLECTION)
    src = run_butler.get('single_visit_star', visit=VISIT, instrument='LSSTCam')
    src_df = src if isinstance(src, pd.DataFrame) else src.asAstropy().to_pandas()
    print(src_df.shape)
    print(list(src_df.columns))

In [ ]:
if RUN_COLLECTION is not None and VISIT is not None:
    idx, sep, ok = mon.crossmatch(src_df['ra'], src_df['dec'], df['ra_deg'], df['dec_deg'], max_sep_arcsec=1.0)
    print(f'matched {ok.sum()} / {len(ok)} sources')

    m_src = mon.flux_njy_to_abmag(src_df['psfFlux'])[ok]
    m_ref = mon.flux_njy_to_abmag(df[f'{PREFIX}_{BAND}_flux'].to_numpy()[idx[ok]])

    fig, ax = plt.subplots()
    ax.plot(m_ref, m_src - m_ref, '.', ms=2, alpha=0.4)
    ax.axhline(0, color='k', lw=1)
    ax.set_ylim(-0.5, 0.5)
    ax.set_xlabel(f'Monster {BAND} (AB mag)')
    ax.set_ylabel('psfFlux mag - Monster mag')
    ax.set_title(f'visit {VISIT}, band {BAND}')
    savefig(fig, f'visit{VISIT}_dmag')